# Combo 4 teacher–student retraining

## Purpose

This notebook trains a compact live-video student from the current multi-scale ViT-L teacher. It is an experiment, not a claim that the student is already superior to the teacher.

## Old architecture

The original live path is `video -> YOLO detector/tracker -> ROI crop -> PraNet refiner -> threshold/overlay`. It is fast, but depends on proposals, processes segmentation frame by frame, and does not explicitly transfer the strongest ViT representation. The earlier research transformer also decoded mainly one final ViT feature, which can lose intermediate spatial detail.

## New architecture

The frozen teacher is `ViT-L/16 -> four transformer-depth taps -> token-to-map reassembly -> residual DPT-style fusion -> full-resolution mask`. The trainable student is `MobileNetV3-small or large -> four CNN feature scales -> 1x1 projections -> top-down fusion -> mask head`. The student is the deployment candidate; the ViT-L remains the accuracy reference.

## Why this design

ViT-L supplies global context and soft spatial supervision but is expensive. MobileNetV3 supplies low latency. Ground-truth Dice/Focal loss protects annotation fidelity, teacher-logit distillation transfers confidence and context, and boundary loss targets small/flat/diffuse lesions. Temporal persistence is evaluated after raw inference because high-confidence false alarms cannot be solved by thresholding alone.

## Novelty boundary

A backbone replacement alone is not novelty. A defensible claim, only if ablations support it, is a benchmark-driven, boundary-aware teacher–student framework for real-time colonoscopy segmentation, evaluated by small/flat/diffuse proxy strata, hard-negative type, domain shift, leakage, and temporal false alarms. The `S_*`, `M_*`, and `B_*` classes are mask-derived proxies, not Paris/NICE/Kudo clinical diagnoses.

## Required comparisons

Run ground-truth-only versus distilled students, without versus with boundary loss, MobileNetV3-small versus large, raw versus temporal-gated outputs, and grouped versus random splits. Keep manifests, seeds, image size, epochs, and optimizer budgets fixed. Do not tune on the canonical test sets.

## Negative-data arm naming

The benchmark site diagnostic resolves `HK20` to `negatives_hyperkvasir_v1`, `PG20` to `negatives_polypgen_v1`, and `MIX20` to the mixed arm used by `r6_mixneg`. Existing ViT-DPT-L MIX20 runs are `r6_mixneg__vit_dpt_l__s42`, `s43`, and `s44`. The decisive CNN arms still missing are xattn and unet_r34 for HK20, PG20, and MIX20. These are experiment labels, not official dataset names; record the manifest, counts, sequence IDs, and hash before publication.

For the full explanation, read `README.md` in the upload folder. Attach this notebook, the ZIP bundle, paired `images/` and `masks/`, a compatible ViT-L checkpoint, and enable a Kaggle GPU.

In [ ]:
from pathlib import Path
import glob, json, os, shutil, subprocess, sys, zipfile

assert Path('/kaggle/input').exists(), 'Enable a Kaggle GPU and attach the input datasets.'
zip_matches = glob.glob('/kaggle/input/**/*.zip', recursive=True)
if not zip_matches:
    raise FileNotFoundError('Attach combo4_teacher_student_retraining.zip as a Kaggle Dataset.')
bundle_zip = next((p for p in zip_matches if 'teacher_student' in Path(p).name.lower()), zip_matches[0])
extract_root = Path('/kaggle/working/combo4_bundle')
if extract_root.exists(): shutil.rmtree(extract_root)
with zipfile.ZipFile(bundle_zip) as archive: archive.extractall(extract_root)
root = extract_root / 'combo4_retraining'
sys.path.insert(0, str(root / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.16', 'opencv-python-headless'], check=True)
print('bundle:', bundle_zip)
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode())


In [ ]:
def find_pair_dirs(root='/kaggle/input'):
    images=[]; masks=[]
    for p in Path(root).rglob('*'):
        if p.is_dir() and p.name.lower() in {'images','image','frames'} and any(p.iterdir()): images.append(p)
        if p.is_dir() and p.name.lower() in {'masks','mask','ground_truth'} and any(p.iterdir()): masks.append(p)
    for image_dir in images:
        for mask_dir in masks:
            if len(list(image_dir.glob('*'))) >= 10 and len(list(mask_dir.glob('*'))) >= 10:
                return image_dir, mask_dir
    raise FileNotFoundError('Attach a dataset with paired images/ and masks/ directories.')

images, masks = find_pair_dirs()
target = root / 'data' / 'kvasir-seg'
target.mkdir(parents=True, exist_ok=True)
shutil.copytree(images, target/'images', dirs_exist_ok=True)
shutil.copytree(masks, target/'masks', dirs_exist_ok=True)
embedded_teachers = [str(p) for p in (root / 'weights').rglob('*.pt')] + [str(p) for p in (root / 'weights').rglob('*.pth')]
attached_teachers = glob.glob('/kaggle/input/**/*.pth', recursive=True) + glob.glob('/kaggle/input/**/*.pt', recursive=True)
teachers = embedded_teachers + attached_teachers
teacher = next((p for p in teachers if 'student' not in p.lower()), '')
teacher_backbone = 'vit_large_patch16_384'
print('images:', len(list((target/'images').glob('*'))), 'masks:', len(list((target/'masks').glob('*'))))
print('teacher:', teacher or 'not found (use ImageNet initialization only for smoke testing)')
print('teacher_backbone:', teacher_backbone)


## Select the retraining runs

The ViT-L teacher stays frozen. Both students use the same split, seed, image size, and loss weights. Set `RUN_LARGE = True` for the primary MobileNetV3-Large candidate; the small run is the latency baseline.

In [ ]:
RUN_LARGE = True
SEED = 42
EPOCHS = 30
student_backbone = 'mobilenetv3_large_100' if RUN_LARGE else 'mobilenetv3_small_100'
out = Path('/kaggle/working') / ('student_large' if RUN_LARGE else 'student_small')
cmd = [sys.executable, str(root/'src/chakra_transformer/train_distill.py'),
       '--teacher-backbone', teacher_backbone,
       '--student-backbone', student_backbone, '--epochs', str(EPOCHS),
       '--batch-size', '8', '--grad-accum', '2', '--workers', '2',
       '--seed', str(SEED), '--output-dir', str(out)]
if teacher: cmd += ['--teacher-checkpoint', teacher]
subprocess.run(cmd, check=True)


In [ ]:
import time, torch
from chakra_transformer.student_segmenter import ChakraLiveStudent
student = ChakraLiveStudent(student_backbone, pretrained=False).cuda().eval()
student.load_state_dict(torch.load(out/'combo4_student_best.pth', map_location='cuda'))
x=torch.randn(1,3,352,352,device='cuda')
for _ in range(20): student(x)
torch.cuda.synchronize(); start=time.perf_counter()
for _ in range(100): student(x)
torch.cuda.synchronize(); elapsed=time.perf_counter()-start
print({'backbone':student_backbone,'parameters':sum(p.numel() for p in student.parameters()),'fps':100/elapsed,'latency_ms':elapsed*10})
shutil.copy(out/'combo4_student_best.pth','/kaggle/working/combo4_student_best.pth')
